In [11]:
# ============================================================
# SECTION 0: ENVIRONMENT SETUP
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import textwrap
import warnings
import re
import os
from collections import Counter

warnings.filterwarnings('ignore')

# Professional matplotlib configuration
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--'
})

# Color palette — historically themed
COLORS = {
    'primary': '#8B4513',
    'secondary': '#CD853F',
    'accent': '#2F4F4F',
    'highlight': '#DAA520',
    'danger': '#B22222',
    'cool': '#4682B4',
    'palette': ['#8B4513', '#CD853F', '#2F4F4F', '#DAA520',
                '#B22222', '#4682B4', '#6B8E23', '#708090']
}

print("✓ Environment configured")

✓ Environment configured


In [14]:
# ============================================================
# SECTION 1: DATA LOADING
# ============================================================

INPUT_DIR = '/kaggle/input'

csv_files = []
for root, dirs, files in os.walk(INPUT_DIR):
    for f in files:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(root, f))

if csv_files:
    DATA_PATH = csv_files[0]
    print(f"✓ Found dataset: {DATA_PATH}")
else:
    DATA_PATH = 'historical_events_tunisia_hannibal.csv'
    print(f"ℹ Using local path: {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH)

print(f"\n{'='*60}")
print(f"  RAW DATA LOADED")
print(f"{'='*60}")
print(f"  Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print(f"  Columns: {list(df_raw.columns)}")
print(f"  Memory: {df_raw.memory_usage(deep=True).sum() / 1024:.1f} KB")

✓ Found dataset: /kaggle/input/historical-dataset-tunisia/hannibal.csv

  RAW DATA LOADED
  Shape: 114 rows × 13 columns
  Columns: ['event_id', 'name_of_incident', 'year', 'country', 'place_name', 'type_of_event', 'historical_character', 'description', 'first_person_strategy', 'mistakes_reflection', 'modern_strategy', 'Unnamed: 11', 'Unnamed: 12']
  Memory: 166.5 KB


In [16]:
# ============================================================
# SECTION 2: REMOVE UNNAMED ARTIFACT COLUMNS
# ============================================================
# WHY: CSV files saved with pandas without index=False create
# residual index columns named 'Unnamed: 0', 'Unnamed: 0.1', etc.
# These are pure artifacts — not real data.
# They pollute metadata, waste memory, and would contaminate
# RAG document construction if left in place.

df = df_raw.copy()

print(f"{'='*60}")
print(f"  STEP 1: UNNAMED COLUMN REMOVAL")
print(f"{'='*60}")
print(f"\n  Columns BEFORE: {list(df.columns)}")
print(f"  Column count: {len(df.columns)}")

# Detect all unnamed columns (case-insensitive)
unnamed_cols = [col for col in df.columns
                if col.lower().startswith('unnamed')]

if unnamed_cols:
    print(f"\n  ⚠ Found {len(unnamed_cols)} artifact column(s):")
    for col in unnamed_cols:
        # Show what's inside to confirm it's just an index
        sample_values = df[col].head(5).tolist()
        nunique = df[col].nunique()
        print(f"    • '{col}' — {nunique} unique values, "
              f"sample: {sample_values}")

    # Drop them
    df = df.drop(columns=unnamed_cols)

    print(f"\n  ✓ Removed: {unnamed_cols}")
    print(f"  Columns AFTER: {list(df.columns)}")
    print(f"  Column count: {len(df.columns)}")
else:
    print(f"\n  ✓ No unnamed artifact columns found — data is clean")

# Update df_raw to match (for consistent audit comparisons later)
df_raw = df_raw.drop(columns=unnamed_cols, errors='ignore')

print(f"\n  Final schema:")
for i, col in enumerate(df.columns, 1):
    print(f"    {i:2d}. {col}")

  STEP 1: UNNAMED COLUMN REMOVAL

  Columns BEFORE: ['event_id', 'name_of_incident', 'year', 'country', 'place_name', 'type_of_event', 'historical_character', 'description', 'first_person_strategy', 'mistakes_reflection', 'modern_strategy']
  Column count: 11

  ✓ No unnamed artifact columns found — data is clean

  Final schema:
     1. event_id
     2. name_of_incident
     3. year
     4. country
     5. place_name
     6. type_of_event
     7. historical_character
     8. description
     9. first_person_strategy
    10. mistakes_reflection
    11. modern_strategy


**NLP Preprocessing **

In [10]:
import pandas as pd
import re
import spacy
import nltk
import numpy as np

from sentence_transformers import SentenceTransformer, util

nltk.download("punkt")
nlp = spacy.load("en_core_web_sm") #for name entity recognition 

# BERT-based sentence embedder
embedder = SentenceTransformer("all-MiniLM-L6-v2")


2026-02-10 08:36:57.295864: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770712617.542186      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770712617.609826      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770712618.189795      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770712618.189838      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770712618.189841      55 computation_placer.cc:177] computation placer alr

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
# lowercase  , remove punctuation and special characters and whitespaces 
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text



In [18]:
#Extract named entities 
def extract_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]


In [19]:
# Preprocess historical records:
# Combine important fields into one text
# Normalize text
# Extract named entities
# Generate semantic embeddings

processed_db = []

for _, row in df.iterrows():

    # Combine structured fields into a semantic text
    combined_text = (
        f"{row['name_of_incident']} "
        f"{row['description']} "
        f"{row['place_name']} "
        f"{row['type_of_event']}"
    )

    # Normalize combined text
    normalized_text = normalize_text(combined_text)

    # Extract named entities (NER)
    entities = extract_entities(combined_text)

    # Generate embedding for semantic search
    embedding = embedder.encode(normalized_text, convert_to_tensor=True)

    # Store processed information
    processed_db.append({
        "name_of_incident": row["name_of_incident"],
        "description": row["description"],
        "place_name": row["place_name"],
        "type_of_event": row["type_of_event"],
        "entities": entities,
        "embedding": embedding
    })


In [26]:
user_query = input("Ask a historical question: ")
print(user_query)
query_normalized = normalize_text(user_query)
print(query_normalized )
query_entities = extract_entities(user_query)
print(query_entities)
query_embedding = embedder.encode(query_normalized, convert_to_tensor=True)
query_embedding

Ask a historical question:  What Happened in Carthage ? 


What Happened in Carthage ? 
what happened in carthage
[]


tensor([ 1.7867e-02,  9.7241e-02,  2.0716e-02,  3.6074e-02,  3.4934e-02,
        -3.3227e-02, -4.4967e-03,  5.0166e-02, -1.2174e-01,  6.7589e-02,
         5.5920e-02, -2.3699e-02,  5.4402e-03,  3.4917e-02, -2.4929e-02,
        -4.0763e-02, -3.0137e-02,  4.2524e-04, -2.7009e-02, -6.4752e-02,
        -5.2967e-03, -2.8239e-02,  1.7299e-03, -8.6784e-03, -4.7457e-02,
         1.8331e-02, -8.9888e-03, -3.8293e-02, -6.8947e-02, -5.4086e-02,
        -8.6891e-02,  1.4035e-02,  4.1679e-02, -1.1057e-01, -3.5620e-03,
         5.3568e-02,  1.1873e-01,  3.8972e-02,  1.4295e-01, -4.2154e-02,
        -1.4412e-02, -3.7731e-02,  4.4930e-02,  3.6714e-02, -4.7113e-02,
         2.2902e-02,  1.0746e-01,  2.1706e-02, -1.0963e-02,  3.5418e-02,
         7.8271e-02,  1.3004e-01,  1.5999e-02,  1.5832e-02, -5.4379e-02,
         2.3166e-02, -4.3404e-02, -5.0464e-02,  7.9171e-02, -6.0823e-02,
         2.3124e-02,  2.9158e-02,  7.6925e-03,  5.2181e-02, -3.4464e-03,
        -1.0218e-02,  8.1569e-02, -2.3287e-02,  1.9

In [6]:
def preprocess_corpus(document, chunk_size=3):
    """
    document: historical text
    chunk_size: number of sentences per chunk
    """
    sentences = sent_tokenize(document)
    chunks = []

    for i in range(0, len(sentences), chunk_size):
        chunk_text = " ".join(sentences[i:i+chunk_size])
        normalized = normalize_text(chunk_text)
        entities = extract_entities(chunk_text)
        hf_tokens = hf_tokenizer.tokenize(chunk_text)

        chunks.append({
            "chunk_text": chunk_text,
            "normalized_text": normalized,
            "entities": entities,
            "hf_tokens": hf_tokens
        })

    return chunks


In [7]:
def preprocess_query(user_query):
    normalized = normalize_text(user_query)
    entities = extract_entities(user_query)
    hf_tokens = hf_tokenizer.tokenize(user_query)

    return {
        "original_query": user_query,
        "normalized_query": normalized,
        "entities": entities,
        "hf_tokens": hf_tokens
    }


In [9]:
user_query = input("Ask a historical question: ")
x= preprocess_query(user_query)
x

Ask a historical question:  what happend in 1950


{'original_query': 'what happend in 1950',
 'normalized_query': 'what happend in 1950',
 'entities': [('1950', 'DATE')],
 'hf_tokens': ['what', 'happen', '##d', 'in', '1950']}